用类装饰器 @delegate('engine') 自动把 Car.start() 调用转发给 Car.engine.start()。

In [1]:
def delegate(attr_name):
    """把未知属性转发给某个实例属性，懒得枚举方法时是首选"""
    def deco(cls):
        def __getattr__(self, item):
            target = getattr(self, attr_name)
            return getattr(target, item)
        cls.__getattr__ = __getattr__
        return cls 
    return deco 

class Engine:
    def start(self):
        print("vroom!")
    def stop(self):
        print("zzz...")


@delegate('engine')
class Car:
    def __init__(self) -> None:
        self.engine = Engine()
        
c = Car()

c.start()
c.stop()       


vroom!
zzz...


In [2]:
def delegate(attr_name, target_type):
    def deco(cls):
        for name, member in target_type.__dict__.items():
            if name.startswith('_') or not callable(member):
                continue
            def make_proxy(method_name):
                def proxy(self, *a, **k):
                    return getattr(getattr(self,attr_name), method_name)(*a, **k)
                return proxy
            setattr(cls, name, make_proxy(name))
            
        return cls 
    return deco 


class Engine:
    def start(self): print("vroom!")
    def stop(self):  print("zzz…")

@delegate('engine', Engine)
class Car:
    def __init__(self):
        self.engine = Engine()

Car().start()     # vroom!
Car().stop()      # zzz…
        
                    

vroom!
zzz…


In [ ]:
# 用@property实现温度转换：
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def fahrenheit(self):
        return self._celsius * 9/5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self._celsius = (value - 32) * 5/9

temp = Temperature(0)
print(temp.fahrenheit)  # 32.0
temp.fahrenheit = 212
print(temp._celsius)    # 100.0